In [4]:
# Cell 1 — imports
import pandas as pd
from pathlib import Path

DATA = Path("../data/ASCII")
list(DATA.glob("*.txt"))

[]

In [20]:
# Cell 2 — first naive read
demo = pd.read_csv(DATA / "DEMO26Q1.txt", sep="$")
demo.shape

/var/folders/l_/wxj_nhwn26x_xwv99jhq6yyh0000gn/T/ipykernel_90499/628982085.py:2: DtypeWarning: Columns (0: to_mfr) have mixed types. Specify dtype option on import or set low_memory=False.
  demo = pd.read_csv(DATA / "DEMO26Q1.txt", sep="$")


(397224, 25)

In [21]:
demo = pd.read_csv(
    DATA / "DEMO26Q1.txt",
    sep="$",
    encoding="latin-1",
    dtype=str,                # read everything as string, we'll type-coerce later
    on_bad_lines="warn",
    keep_default_na=False,    # don't auto-convert "NA"/"" to NaN; we want to see what's there
    na_values=[""],           # but treat truly empty strings as NaN
)
demo.shape

(397224, 25)

In [22]:
demo.head()

,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,...,age_grp,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country
0,1001678127,10016781,27,F,20120330,20260121,20140318,20260203,EXP,NaN,...,NaN,F,Y,NaN,NaN,20260203,NaN,HP,CA,CA
1,100293665,10029366,5,F,NaN,20251229,20140322,20260106,EXP,NaN,...,NaN,M,Y,NaN,NaN,20260106,NaN,HP,AU,AU
2,1005450711,10054507,11,F,NaN,20260114,20140402,20260120,PER,NaN,...,E,F,Y,NaN,NaN,20260120,NaN,MD,US,US
3,1005762126,10057621,26,F,20140204,20260112,20140404,20260127,EXP,NaN,...,A,M,Y,53,KG,20260127,NaN,MD,CA,CA
4,100670805,10067080,5,F,NaN,20140428,20260119,20260119,PER,NaN,...,NaN,M,Y,NaN,NaN,20260119,NaN,MD,TR,US


In [23]:
demo["to_mfr"].value_counts(dropna=False)

to_mfr
NaN    384965
N       10891
Y        1368
Name: count, dtype: int64

In [24]:
for col in demo.columns:
    print(f"\n--- {col} (nulls: {demo[col].isna().sum()}) ---")
    print(demo[col].value_counts(dropna=False).head(10))


--- primaryid (nulls: 0) ---
primaryid
1001678127    1
100293665     1
1005450711    1
1005762126    1
100670805     1
100716524     1
1007844154    1
101092286     1
1017130217    1
1017572234    1
Name: count, dtype: int64

--- caseid (nulls: 0) ---
caseid
10016781    1
10029366    1
10054507    1
10057621    1
10067080    1
10071652    1
10078441    1
10109228    1
10171302    1
10175722    1
Name: count, dtype: int64

--- caseversion (nulls: 0) ---
caseversion
1     267307
2      80192
3      24044
4      10407
5       4930
6       2827
7       1813
8       1184
9        871
10       669
Name: count, dtype: int64

--- i_f_code (nulls: 0) ---
i_f_code
I    267307
F    129917
Name: count, dtype: int64

--- event_dt (nulls: 229258) ---
event_dt
NaN         229258
20250101     15599
20260101     10176
20251201      5933
20240101      4851
20260201      4205
20251101      2854
20251001      2206
20250901      1631
20230101      1497
Name: count, dtype: int64

--- mfr_dt (nulls: 0) ---


In [25]:
for col in ["age", "wt"]:
    s = pd.to_numeric(demo[col], errors="coerce")
    print(f"\n--- {col} ---")
    print(s.describe())
    print(f"  values that failed to parse: {(s.isna() & demo[col].notna()).sum()}")


--- age ---
count    245616.000000
mean         54.272893
std          23.894782
min           0.000000
25%          40.000000
50%          59.000000
75%          71.000000
max        1043.000000
Name: age, dtype: float64
  values that failed to parse: 0

--- wt ---
count    69091.000000
mean        75.695322
std        323.988017
min          0.000000
25%         58.900000
50%         71.667000
75%         87.430000
max      80505.000000
Name: wt, dtype: float64
  values that failed to parse: 0


In [27]:
# Convert with unit awareness
age_num = pd.to_numeric(demo["age"], errors="coerce")

# Roughly normalise to years for inspection
unit_to_years = {"YR": 1, "DEC": 10, "MON": 1/12, "WK": 1/52, "DY": 1/365.25, "HR": 1/8766}
age_years = age_num * demo["age_cod"].map(unit_to_years)

# Suspicious ages
print("Implausibly old (>120 years equivalent):")
print(demo[age_years > 120][["primaryid", "age", "age_cod"]].head(20))
print(f"\nTotal implausible: {(age_years > 120).sum()}")

# Weight outliers
wt_num = pd.to_numeric(demo["wt"], errors="coerce")
print("\nImplausibly heavy (>500 kg, assuming KG):")
print(demo[(wt_num > 500) & (demo["wt_cod"] == "KG")][["primaryid", "wt", "wt_cod"]].head(20))

# How many sub-1kg "weights"?
print(f"\nWeights <1: {(wt_num < 1).sum()} rows")
print(f"Weights <10: {(wt_num < 10).sum()} rows")

Implausibly old (>120 years equivalent):
        primaryid  age age_cod
137615  262740041   23     DEC
244150  263899411  126      YR
348641  265065331   40     DEC
350193  265083411   70     DEC

Total implausible: 4

Implausibly heavy (>500 kg, assuming KG):
        primaryid      wt wt_cod
34153   248762252  551.15     KG
173467  263128201    4105     KG
237201  263822571   80505     KG
237612  263827301   26775     KG
343194  265001761     849     KG

Weights <1: 64 rows
Weights <10: 937 rows


In [28]:
# Cross-check: are the 4 implausible-age rows DEC-coded? And is the lone YR one really 126?
demo.loc[demo["primaryid"].astype(str).isin(["262740041", "265065331", "265083411"]), 
         ["primaryid", "age", "age_cod", "sex", "age_grp", "occr_country"]]

,primaryid,age,age_cod,sex,age_grp,occr_country
137615,262740041,23,DEC,M,A,US
348641,265065331,40,DEC,F,NaN,US
350193,265083411,70,DEC,NaN,NaN,AU
